In [1]:
import os
import gc
import json
import math
import copy
import random
import hashlib
import warnings
import inspect
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Iterable

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

import matplotlib.pyplot as plt

from scipy import stats

from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold, train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:

# Path setup and notebook configuration

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if (cand / "artifacts").exists() and (cand / "data").exists():
            return cand
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
ARTIFACTS = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"

NB11_REPORT_ROOT = ARTIFACTS / "reports" / "notebook 11"
NB11_META_ROOT = ARTIFACTS / "metadata" / "notebook 11"
NB11_CACHE_ROOT = ARTIFACTS / "cache" / "notebook 11"
SHARED_META = NB11_META_ROOT / "_shared"
MODE_COMPARE_ROOT = NB11_REPORT_ROOT / "mode_comparison"

for p in [NB11_REPORT_ROOT, NB11_META_ROOT, NB11_CACHE_ROOT, SHARED_META, MODE_COMPARE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

RUN_MODES = ["pca", "pathway"]

FEATURE_REPR_MODE = None
USE_PATHWAY_FEATURES = False
MODE_TAG = None

OUT_ROOT = None
OUT_RESULTS = None
OUT_FIGS = None
OUT_INTERP = None
OUT_COMPARE = None
OUT_META = None
OUT_CACHE = None
CACHE_VERSION = None

def set_run_mode(mode: str) -> None:
    global FEATURE_REPR_MODE, USE_PATHWAY_FEATURES, MODE_TAG
    global OUT_ROOT, OUT_RESULTS, OUT_FIGS, OUT_INTERP, OUT_COMPARE, OUT_META, OUT_CACHE, CACHE_VERSION

    mode = str(mode).strip().lower()
    if mode not in RUN_MODES:
        raise ValueError(f"mode must be one of {RUN_MODES}, got {mode!r}")

    FEATURE_REPR_MODE = mode
    USE_PATHWAY_FEATURES = FEATURE_REPR_MODE == "pathway"
    MODE_TAG = FEATURE_REPR_MODE

    OUT_ROOT = NB11_REPORT_ROOT / MODE_TAG
    OUT_RESULTS = OUT_ROOT / "results"
    OUT_FIGS = OUT_ROOT / "figures"
    OUT_INTERP = OUT_ROOT / "interpretability"
    OUT_COMPARE = OUT_ROOT / "baseline_comparison"
    OUT_META = NB11_META_ROOT / MODE_TAG
    OUT_CACHE = NB11_CACHE_ROOT / MODE_TAG

    for p in [OUT_ROOT, OUT_RESULTS, OUT_FIGS, OUT_INTERP, OUT_COMPARE, OUT_META, OUT_CACHE]:
        p.mkdir(parents=True, exist_ok=True)

    CACHE_VERSION = f"gated_cross_modal_attention_notebook11_{MODE_TAG}_v1"

    print("Project root:", PROJECT_ROOT)
    print("Notebook 11 mode:", FEATURE_REPR_MODE)
    print("Notebook 11 output root:", OUT_ROOT)

SEEDS = [19537, 1584678, 17052356]
PRIMARY_TARGET = "auc"
TOP_K_DRUGS = 10
N_SPLITS_DESIRED = 10
MIN_CELLS_PER_DRUG = 60
VAL_FRACTION = 0.20

ARM_THRESHOLDS = {
    "prot_procan_depmapSanger": 0.40,
    "prot_ms_ccle_gygi": 0.30,
    "prot_combined_union": 0.60,
    "prot_rppa_ccle": 0.50,
}

FEATURE_SETS = [
    "rna",
    "cnv",
    "mut",
    "prot",
    "rna+cnv",
    "rna+mut",
    "rna+prot",
    "cnv+mut",
    "cnv+prot",
    "mut+prot",
    "rna+cnv+mut",
    "rna+cnv+prot",
    "rna+mut+prot",
    "cnv+mut+prot",
    "rna+cnv+mut+prot",
]
FEATURE_SET_MODALITIES = {fs: tuple(fs.split("+")) for fs in FEATURE_SETS}
ALL_MODALITIES = ["rna", "cnv", "mut", "prot"]

PCA_CONFIG = {
    "continuous_components": 256,
    "mutation_top_k": 1024,
    "mutation_svd_components": 128,
    "min_train_rows_for_pca": 8,
}

MODEL_CONFIG = {
    "tower_hidden_dim": 256,
    "tower_out_dim": 128,
    "fusion_heads": 4,
    "dropout": 0.20,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "max_epochs": 80,
    "patience": 12,
    "min_delta": 1e-4,
    "batch_size": 64,
    "num_workers": 0,
    "grad_clip_norm": 5.0,
    "track2_modality_dropout": 0.10,
    "loss": "smooth_l1",
}

ARM_BATCH_SIZE = {
    "prot_combined_union": 32,
    "prot_procan_depmapSanger": 64,
    "prot_ms_ccle_gygi": 64,
    "prot_rppa_ccle": 64,
}

FAST_DEV_RUN = False
FAST_DEV_LIMIT_TRACKS = None
FAST_DEV_LIMIT_ARMS = None
FAST_DEV_LIMIT_FEATURES = None
FAST_DEV_LIMIT_DRUGS = None
FORCE_RERUN = False

RESUME_FROM_CHECKPOINT = True
USE_SPLIT_CACHE = True
USE_FEATURE_CACHE = True
USE_MODEL_STATE_RESUME = True
WRITE_CACHE_METADATA = True
CHECKPOINT_EVERY_N_ROWS = 50
CHECKPOINT_EVERY_N_DRUGS = 5
TRAIN_STATE_SAVE_EVERY_EPOCHS = 10

def get_batch_size_for_arm(arm_name: str) -> int:
    return int(ARM_BATCH_SIZE.get(arm_name, MODEL_CONFIG["batch_size"]))

In [3]:

# Input paths and reproducibility block

PATH_CANDIDATES = {
    "cell_index": [
        ARTIFACTS / "cleaned" / "notebook 2" / "cell_index.parquet",
        ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "cell_metadata.parquet",
        ARTIFACTS / "metadata" / "notebook 1" / "cell_metadata_depmap.parquet",
    ],
    "prism_long": [
        ARTIFACTS / "cleaned" / "notebook 2" / "prism_long.parquet",
    ],
    "gene_index": [
        ARTIFACTS / "cleaned" / "notebook 2" / "gene_index.parquet",
    ],
    "rna": [
        ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "rna.parquet",
        DATA_DIR / "depmap" / "processed" / "Expression_Public_25Q3_subsetted.csv",
    ],
    "cnv": [
        ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "cnv.parquet",
        DATA_DIR / "depmap" / "processed" / "Copy_Number_WGS_Public_25Q3_subsetted.csv",
    ],
    "mut": [
        ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "mut.parquet",
        DATA_DIR / "depmap" / "processed" / "Damaging_Mutations_subsetted.csv",
    ],
    "prot_procan_depmapSanger": [
        ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "prot_optional__prot_procan_depmapSanger.parquet",
    ],
    "prot_ms_ccle_gygi": [
        ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "prot_optional__prot_ms_ccle_gygi.parquet",
    ],
    "prot_combined_union": [
        ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "prot_optional__prot_combined_union.parquet",
    ],
    "prot_rppa_ccle": [
        ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "prot_optional__prot_rppa_ccle.parquet",
    ],
}

PATHWAY_GMT = DATA_DIR / "pathways" / "ReactomePathways.gmt",

TOP_DRUG_CANDIDATE_PATHS = [
    ARTIFACTS / "metadata" / "notebook 10" / "top10_drugs.json",
    ARTIFACTS / "metadata" / "notebook 11" / "top10_drugs.json",
    ARTIFACTS / "reports" / "notebook 9" / "top10_drugs.json",
    ARTIFACTS / "reports" / "notebook 8" / "top10_drugs.json",
]

BASELINE_SCAN_DIRS = [
    ARTIFACTS / "reports" / "notebook 3b",
    ARTIFACTS / "reports" / "notebook 4",
    ARTIFACTS / "reports" / "notebook 5",
    ARTIFACTS / "reports" / "notebook 6",
    ARTIFACTS / "reports" / "notebook 7",
    ARTIFACTS / "reports" / "notebook 8",
    ARTIFACTS / "reports" / "notebook 9",
    ARTIFACTS / "reports" / "notebook 10",
]

BASE_SEED = SEEDS[0]
SEED = BASE_SEED
RNG = np.random.default_rng(SEED)

def set_all_seeds(seed: int) -> None:
    global SEED, RNG
    SEED = int(seed)
    RNG = np.random.default_rng(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    os.environ["PYTHONHASHSEED"] = str(SEED)

set_all_seeds(BASE_SEED)

if torch.cuda.is_available():
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

SHARED_RUN_METADATA = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "device": str(DEVICE),
    "seeds": SEEDS,
    "target": PRIMARY_TARGET,
    "top_k_drugs": TOP_K_DRUGS,
    "n_splits_desired": N_SPLITS_DESIRED,
    "min_cells_per_drug": MIN_CELLS_PER_DRUG,
    "feature_sets": FEATURE_SETS,
    "run_modes": RUN_MODES,
    "arm_thresholds": ARM_THRESHOLDS,
    "pca_config": PCA_CONFIG,
    "model_config": MODEL_CONFIG,
}
with open(SHARED_META / "run_metadata_shared.json", "w") as f:
    json.dump(SHARED_RUN_METADATA, f, indent=2)

Using device: cuda


In [4]:

# Helper utilities

def first_existing(paths: Iterable[Path]) -> Optional[Path]:
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix in {".csv", ".txt", ".tsv"}:
        sep = "\t" if path.suffix in {".tsv", ".txt"} else ","
        return pd.read_csv(path, sep=sep, engine="python")
    raise ValueError(f"Unsupported table format: {path}")

def sha256_file(path: Path, block_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def describe_file(path: Optional[Path]) -> Optional[dict]:
    if path is None:
        return None
    return {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "mtime": datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).isoformat(),
    }

def detect_column(df: pd.DataFrame, candidates: List[str], required: bool = True) -> Optional[str]:
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise KeyError(f"Could not find required column among candidates: {candidates}")
    return None

def coerce_depmap_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "depmap_id" in df.columns:
        df = df.set_index("depmap_id")
    elif df.index.name != "depmap_id":
        for cand in ["DepMap_ID", "DepMapID", "depmapId"]:
            if cand in df.columns:
                df = df.set_index(cand)
                break
    df.index = df.index.astype(str)
    df.index.name = "depmap_id"
    return df

def safe_numeric_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def to_float32_df(df: pd.DataFrame) -> pd.DataFrame:
    return df.astype(np.float32)

def stable_hash(obj, n: int = 16) -> str:
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()[:n]

def safe_slug(x: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x)).strip("_")

def atomic_write_json(obj, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)

def atomic_to_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=index)
    os.replace(tmp, path)

def atomic_to_parquet(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)

def write_cache_metadata(path: Path, meta: dict) -> None:
    if not WRITE_CACHE_METADATA:
        return
    atomic_write_json(meta, path)

def load_json_if_exists(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def update_run_state(path: Path, updates: dict) -> dict:
    state = load_json_if_exists(path, default={}) or {}
    state.update(updates)
    state["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
    atomic_write_json(state, path)
    return state

def resolve_group_column(cell_index: pd.DataFrame) -> str:
    for cand in ["lineage_1", "primary_disease", "lineage_2", "lineage_3"]:
        if cand in cell_index.columns:
            return cand
    raise KeyError("No lineage-like grouping column found in cell_index.")

def spearman_safe(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return np.nan
    if np.allclose(y_true, y_true[0]) or np.allclose(y_pred, y_pred[0]):
        return np.nan
    return float(stats.spearmanr(y_true, y_pred, nan_policy="omit").statistic)

def r2_safe(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return np.nan
    try:
        return float(r2_score(y_true, y_pred))
    except Exception:
        return np.nan

def canonicalise_gene_token(x: str) -> Optional[str]:
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    s = s.split("::")[-1]
    for pref in ["rna__", "cnv__", "mut__", "prot__", "ms__", "rppa__", "procan__"]:
        if s.lower().startswith(pref):
            s = s[len(pref):]
    s = s.strip().upper()
    s = re.sub(r"\(.*?\)", "", s)
    s = re.split(r"[|;,/ ]", s)[0]
    s = re.split(r"[_:]", s)[0]
    s = re.sub(r"[^A-Z0-9\-]", "", s)
    if s in {"", "NA", "NAN", "NONE"}:
        return None
    if s.startswith("ENSG"):
        return None
    return s

def find_first_existing_path(paths: List[Path]) -> Optional[Path]:
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

In [5]:

# Load inputs and align the common backbone

input_registry = {}
resolved_paths = {}
for key, candidates in PATH_CANDIDATES.items():
    resolved = first_existing(candidates)
    resolved_paths[key] = resolved
    input_registry[key] = describe_file(resolved)

with open(SHARED_META / "resolved_input_paths.json", "w") as f:
     json.dump({k: (str(v) if v is not None else None) for k, v in resolved_paths.items()}, f, indent=2)

print("Resolved inputs:")
display(pd.DataFrame([
    {"name": k, "path": None if v is None else str(v)} for k, v in resolved_paths.items()
]))

missing_required = [k for k in ["cell_index", "prism_long", "rna", "cnv", "mut"] if resolved_paths[k] is None]
if missing_required:
    raise FileNotFoundError(f"Missing required inputs: {missing_required}")

cell_index = read_table(resolved_paths["cell_index"])
if "depmap_id" not in cell_index.columns and cell_index.index.name != "depmap_id":
    cell_index = coerce_depmap_index(cell_index).reset_index()
cell_index["depmap_id"] = cell_index["depmap_id"].astype(str)
cell_index = cell_index.drop_duplicates("depmap_id").set_index("depmap_id")
group_col = resolve_group_column(cell_index)

prism_long = read_table(resolved_paths["prism_long"])
if "depmap_id" not in prism_long.columns:
    depmap_col = detect_column(prism_long, ["DepMap_ID", "depmapId", "depmap_id"])
    prism_long = prism_long.rename(columns={depmap_col: "depmap_id"})
prism_long["depmap_id"] = prism_long["depmap_id"].astype(str)

target_col = detect_column(prism_long, ["target"])
value_col = detect_column(prism_long, ["y", "value", "response"])
compound_col = detect_column(prism_long, ["compound_id", "drug_id", "compound"])

prism_long = prism_long.rename(columns={
    target_col: "target",
    value_col: "y",
    compound_col: "compound_id",
})
prism_long["compound_id"] = prism_long["compound_id"].astype(str)
prism_long["y"] = pd.to_numeric(prism_long["y"], errors="coerce")

prism_auc = prism_long[prism_long["target"].astype(str).str.lower() == PRIMARY_TARGET].copy()
prism_auc = prism_auc[prism_auc["depmap_id"].isin(cell_index.index)].copy()

def load_modality_table(key: str) -> pd.DataFrame:
    path = resolved_paths[key]
    if path is None:
        raise FileNotFoundError(f"No path resolved for {key}")
    df = read_table(path)
    df = coerce_depmap_index(df)
    df = safe_numeric_frame(df)
    return to_float32_df(df)

rna_df = load_modality_table("rna")
cnv_df = load_modality_table("cnv")
mut_df = load_modality_table("mut").fillna(0.0)

common_backbone = sorted(set(cell_index.index) & set(rna_df.index) & set(cnv_df.index) & set(mut_df.index))
cell_index = cell_index.loc[common_backbone].copy()
rna_df = rna_df.loc[common_backbone]
cnv_df = cnv_df.loc[common_backbone]
mut_df = mut_df.loc[common_backbone]
prism_auc = prism_auc[prism_auc["depmap_id"].isin(common_backbone)].copy()

print("Backbone cohort size:", len(common_backbone))
print("AUC rows in prism_long:", prism_auc.shape[0])

gene_index = None
if resolved_paths["gene_index"] is not None:
    gene_index = read_table(resolved_paths["gene_index"])
    print("Loaded gene_index:", gene_index.shape)
    print("gene_index columns:", gene_index.columns.tolist())

Resolved inputs:


,name,path
0,cell_index,/home/andrija/Desktop/Final Year Project/FYP/a...
1,prism_long,/home/andrija/Desktop/Final Year Project/FYP/a...
2,gene_index,/home/andrija/Desktop/Final Year Project/FYP/a...
3,rna,/home/andrija/Desktop/Final Year Project/FYP/a...
4,cnv,/home/andrija/Desktop/Final Year Project/FYP/a...
5,mut,/home/andrija/Desktop/Final Year Project/FYP/a...
6,prot_procan_depmapSanger,/home/andrija/Desktop/Final Year Project/FYP/a...
7,prot_ms_ccle_gygi,/home/andrija/Desktop/Final Year Project/FYP/a...
8,prot_combined_union,/home/andrija/Desktop/Final Year Project/FYP/a...
9,prot_rppa_ccle,/home/andrija/Desktop/Final Year Project/FYP/a...


Backbone cohort size: 1079
AUC rows in prism_long: 306903
Loaded gene_index: (96689, 7)
gene_index columns: ['feature_name', 'feature_base', 'platform_prefix', 'modality', 'source', 'gene_symbol_canonical', 'canonical_changed']


In [6]:

# Optional gene lookup and pathway resources

def build_gene_lookup(gene_index: Optional[pd.DataFrame]) -> Dict[str, str]:
    if gene_index is None or gene_index.empty:
        print("[INFO] gene_index missing or empty; pathway mode will be unavailable.")
        return {}

    gi = gene_index.copy()
    cols = {str(c).lower(): c for c in gi.columns}
    gene_col = cols.get("gene_symbol_canonical")
    if gene_col is None:
        print("[INFO] gene_symbol_canonical not found; pathway mode will be unavailable.")
        return {}

    key_cols = []
    for cand in ["feature_name", "feature_base"]:
        col = cols.get(cand)
        if col is not None:
            key_cols.append(col)

    if not key_cols:
        print("[INFO] No usable feature key columns found in gene_index; pathway mode will be unavailable.")
        return {}

    gi[gene_col] = gi[gene_col].map(canonicalise_gene_token)
    gi = gi.dropna(subset=[gene_col]).copy()

    lookup = {}
    for key_col in key_cols:
        tmp = gi[[key_col, gene_col]].copy()
        tmp[key_col] = tmp[key_col].astype(str).str.strip()
        tmp = tmp[tmp[key_col] != ""].drop_duplicates(subset=[key_col], keep="first")
        lookup.update(dict(zip(tmp[key_col], tmp[gene_col])))

    print(f"Built gene lookup with {len(lookup)} entries.")
    return lookup

def feature_to_gene_map(columns: Iterable[str], gene_lookup: Dict[str, str]) -> Dict[str, str]:
    mapping = {}
    for col in columns:
        col = str(col)
        gene = gene_lookup.get(col)
        if gene is None:
            gene = canonicalise_gene_token(col)
        if gene is not None:
            mapping[col] = gene
    return mapping

def aggregate_features_to_genes(df: pd.DataFrame, mapping: Dict[str, str]) -> pd.DataFrame:
    keep_cols = [c for c in df.columns if c in mapping]
    if not keep_cols:
        return pd.DataFrame(index=df.index)
    tmp = df[keep_cols].rename(columns=mapping)
    out = tmp.T.groupby(level=0).mean().T
    out.index = df.index.astype(str)
    out.index.name = "depmap_id"
    return to_float32_df(out)

def load_gmt(path: Path) -> Dict[str, set]:
    gene_sets = {}
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            name = parts[0]
            genes = {canonicalise_gene_token(g) for g in parts[2:]}
            genes = {g for g in genes if g is not None}
            if genes:
                gene_sets[name] = genes
    return gene_sets

GENE_LOOKUP = build_gene_lookup(gene_index)
GMT_PATH = find_first_existing_path(PATHWAY_GMT)
ALL_GENE_SETS = load_gmt(GMT_PATH) if GMT_PATH is not None else {}

if "pathway" in RUN_MODES and GMT_PATH is None:
    raise FileNotFoundError("RUN_MODES includes 'pathway' but no GMT file was found.")

if GMT_PATH is not None:
    print("Found pathway GMT:", GMT_PATH)
    print("Loaded gene sets:", len(ALL_GENE_SETS))

Built gene lookup with 40190 entries.
Found pathway GMT: /home/andrija/Desktop/Final Year Project/FYP/data/pathways/ReactomePathways.gmt
Loaded gene sets: 2855


In [7]:

# Resolve the top-K drugs

def try_load_top_drugs_from_json(path: Path) -> Optional[List[str]]:
    if not path.exists():
        return None
    with open(path, "r") as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        for key in ["top10_drugs", "top_drugs", "compound_ids", "drugs"]:
            if key in obj and isinstance(obj[key], list) and len(obj[key]) >= TOP_K_DRUGS:
                return [str(x) for x in obj[key][:TOP_K_DRUGS]]
    if isinstance(obj, list) and len(obj) >= TOP_K_DRUGS:
        return [str(x) for x in obj[:TOP_K_DRUGS]]
    return None

resolved_top_drugs = None
for p in TOP_DRUG_CANDIDATE_PATHS:
    maybe = try_load_top_drugs_from_json(p)
    if maybe is not None:
        resolved_top_drugs = maybe
        print("Loaded fixed top drug list from:", p)
        break

if resolved_top_drugs is None:
    coverage = prism_auc.groupby("compound_id")["depmap_id"].nunique().sort_values(ascending=False)
    resolved_top_drugs = coverage.head(TOP_K_DRUGS).index.astype(str).tolist()
    print("Derived top drugs from AUC coverage in prism_long.")

TOP_DRUGS = resolved_top_drugs[:TOP_K_DRUGS]
atomic_write_json({"top10_drugs": TOP_DRUGS}, SHARED_META / "top10_drugs.json")
display(pd.DataFrame({"compound_id": TOP_DRUGS}))

Derived top drugs from AUC coverage in prism_long.


,compound_id
0,IXAZOMIB (BRD:BRD-K78659596-001-03-9)
1,OTS167 (BRD:BRD-K53417444-003-03-1)
2,SB-2343 (BRD:BRD-K98795921-001-01-7)
3,PF-05212384 (BRD:BRD-K07955840-001-02-3)
4,A-1065-5 (BRD:BRD-K87124298-001-03-4)
5,CR8-(R) (BRD:BRD-K40331046-305-01-5)
6,BERZOSERTIB (BRD:BRD-K04701033-001-03-9)
7,TG-02 (BRD:BRD-K14560436-001-01-4)
8,RUBITECAN (BRD:BRD-K79821389-001-03-5)
9,PI3K-IN-2 (BRD:BRD-K62374002-001-01-5)


In [8]:

# Load and threshold all proteomics arms, then build arm-aware modality tables

def load_and_prepare_proteomics_arm(arm_name: str, threshold: float) -> dict:
    path = resolved_paths[arm_name]
    if path is None:
        raise FileNotFoundError(f"Missing proteomics path for {arm_name}")

    raw = load_modality_table(arm_name)
    raw = raw.loc[raw.index.intersection(common_backbone)].copy()

    arm_observed_mask = raw.notna().any(axis=1)
    raw_arm = raw.loc[arm_observed_mask].copy()

    if raw_arm.empty:
        empty_report = pd.DataFrame(columns=["raw_feature", "kept_by_threshold", "missing_fraction_arm_observed", "mapped_gene"])
        empty_report.to_csv(SHARED_META / f"{arm_name}__feature_threshold_report.csv", index=False)
        return {
            "raw": raw,
            "raw_arm": raw_arm,
            "raw_kept": pd.DataFrame(index=raw.index),
            "gene_df": pd.DataFrame(index=raw.index),
            "eligible_cells": [],
            "meta": {
                "arm": arm_name,
                "threshold": threshold,
                "n_cells_raw": int(raw.shape[0]),
                "n_cells_arm_observed": 0,
                "n_features_raw": int(raw.shape[1]),
                "n_features_kept": 0,
                "n_genes_after_aggregation": 0,
                "n_eligible_cells": 0,
                "cell_coverage_fraction": 0.0,
                "overall_missing_after_threshold": np.nan,
            },
        }

    missing_frac = raw_arm.isna().mean(axis=0)
    keep = missing_frac <= threshold

    raw_kept = raw.loc[:, keep].copy()
    raw_kept_arm = raw_arm.loc[:, keep].copy()

    gene_mapping = feature_to_gene_map(raw_kept.columns, GENE_LOOKUP)
    gene_df = aggregate_features_to_genes(raw_kept, gene_mapping)

    eligible_cells = sorted(raw_kept.index[raw_kept.notna().any(axis=1)].astype(str).tolist())

    feature_report = pd.DataFrame({
        "raw_feature": raw.columns.astype(str),
        "kept_by_threshold": raw.columns.isin(raw_kept.columns),
        "missing_fraction_arm_observed": raw_arm.isna().mean(axis=0).reindex(raw.columns).values,
        "mapped_gene": [gene_mapping.get(c) for c in raw.columns.astype(str)],
    })
    feature_report.to_csv(SHARED_META / f"{arm_name}__feature_threshold_report.csv", index=False)

    meta = {
        "arm": arm_name,
        "threshold": threshold,
        "n_cells_raw": int(raw.shape[0]),
        "n_cells_arm_observed": int(raw_arm.shape[0]),
        "n_features_raw": int(raw.shape[1]),
        "n_features_kept": int(raw_kept.shape[1]),
        "n_genes_after_aggregation": int(gene_df.shape[1]),
        "n_eligible_cells": int(len(eligible_cells)),
        "cell_coverage_fraction": float(len(eligible_cells) / max(len(common_backbone), 1)),
        "overall_missing_after_threshold": float(raw_kept_arm.isna().mean().mean()) if raw_kept_arm.shape[1] else np.nan,
    }

    return {
        "raw": raw,
        "raw_arm": raw_arm,
        "raw_kept": raw_kept,
        "gene_df": gene_df,
        "eligible_cells": eligible_cells,
        "meta": meta,
    }

prot_arm_data = {}
for arm_name, thr in ARM_THRESHOLDS.items():
    prot_arm_data[arm_name] = load_and_prepare_proteomics_arm(arm_name, thr)

for arm_name in ARM_THRESHOLDS:
    arm_obj = prot_arm_data[arm_name]
    print(
        arm_name,
        "| raw:", arm_obj["raw"].shape,
        "| raw_kept:", arm_obj["raw_kept"].shape,
        "| gene_df:", arm_obj["gene_df"].shape,
        "| eligible_cells:", len(arm_obj["eligible_cells"]),
    )

arm_meta_df = pd.DataFrame([v["meta"] for v in prot_arm_data.values()]).sort_values("arm").reset_index(drop=True)
arm_meta_df.to_csv(SHARED_META / "proteomics_arm_threshold_metadata.csv", index=False)
display(arm_meta_df)

rna_gene_df = aggregate_features_to_genes(rna_df, feature_to_gene_map(rna_df.columns, GENE_LOOKUP))
cnv_gene_df = aggregate_features_to_genes(cnv_df, feature_to_gene_map(cnv_df.columns, GENE_LOOKUP))
mut_gene_df = aggregate_features_to_genes(mut_df, feature_to_gene_map(mut_df.columns, GENE_LOOKUP))

prot_procan_depmapSanger | raw: (1079, 7906) | raw_kept: (1079, 4647) | gene_df: (1079, 4647) | eligible_cells: 485
prot_ms_ccle_gygi | raw: (1079, 11780) | raw_kept: (1079, 7880) | gene_df: (1079, 7880) | eligible_cells: 304
prot_combined_union | raw: (1079, 18751) | raw_kept: (1079, 11524) | gene_df: (1079, 7110) | eligible_cells: 679
prot_rppa_ccle | raw: (1079, 144) | raw_kept: (1079, 144) | gene_df: (1079, 144) | eligible_cells: 612


,arm,threshold,n_cells_raw,n_cells_arm_observed,n_features_raw,n_features_kept,n_genes_after_aggregation,n_eligible_cells,cell_coverage_fraction,overall_missing_after_threshold
0,prot_combined_union,0.6,1079,679,18751,11524,7110,679,0.629286,0.471390
1,prot_ms_ccle_gygi,0.3,1079,304,11780,7880,7880,304,0.281742,0.041813
2,prot_procan_depmapSanger,0.4,1079,485,7906,4647,4647,485,0.449490,0.100472
3,prot_rppa_ccle,0.5,1079,612,144,144,144,612,0.567192,0.000000


In [9]:

# Arm-aware base modality frames and sample-presence masks

def modality_frame_for_arm(modality: str, arm_name: str) -> pd.DataFrame:
    if modality == "rna":
        return rna_df
    if modality == "cnv":
        return cnv_df
    if modality == "mut":
        return mut_df
    if modality == "prot":
        return prot_arm_data[arm_name]["raw_kept"]
    raise KeyError(modality)

def modality_gene_frame_for_arm(modality: str, arm_name: str) -> pd.DataFrame:
    if modality == "rna":
        return rna_gene_df
    if modality == "cnv":
        return cnv_gene_df
    if modality == "mut":
        return mut_gene_df
    if modality == "prot":
        return prot_arm_data[arm_name]["gene_df"]
    raise KeyError(modality)

def sample_presence_for_frame(df: pd.DataFrame) -> pd.Series:
    if df.empty:
        return pd.Series(0.0, index=df.index, dtype=np.float32)
    return df.notna().any(axis=1).astype(np.float32)

presence_by_arm = {}
missing_summary_rows = []

for arm_name in ARM_THRESHOLDS:
    base_index = pd.Index(prot_arm_data[arm_name]["eligible_cells"], name="depmap_id")
    tmp = pd.DataFrame(index=base_index)
    for modality in ALL_MODALITIES:
        frame = modality_frame_for_arm(modality, arm_name).reindex(index=base_index)
        tmp[modality] = sample_presence_for_frame(frame)
        missing_summary_rows.append({
            "arm": arm_name,
            "modality": modality,
            "n_cells_arm_base": int(len(base_index)),
            "present_fraction": float(tmp[modality].mean()) if len(tmp) else np.nan,
        })
    presence_by_arm[arm_name] = tmp.fillna(0.0)

presence_summary_df = pd.DataFrame(missing_summary_rows)
presence_summary_df.to_csv(SHARED_META / "arm_modality_presence_summary.csv", index=False)
display(presence_summary_df)

,arm,modality,n_cells_arm_base,present_fraction
0,prot_procan_depmapSanger,rna,485,1.0
1,prot_procan_depmapSanger,cnv,485,1.0
2,prot_procan_depmapSanger,mut,485,1.0
3,prot_procan_depmapSanger,prot,485,1.0
4,prot_ms_ccle_gygi,rna,304,1.0
5,prot_ms_ccle_gygi,cnv,304,1.0
6,prot_ms_ccle_gygi,mut,304,1.0
7,prot_ms_ccle_gygi,prot,304,1.0
8,prot_combined_union,rna,679,1.0
9,prot_combined_union,cnv,679,1.0


In [10]:

# Fold-safe split helpers

def safe_group_folds(sample_ids: List[str], groups: pd.Series, seed: int, n_splits_desired: int = N_SPLITS_DESIRED):
    sample_ids = np.asarray(sample_ids, dtype=object)
    groups = groups.loc[sample_ids].astype(str)
    perm_rng = np.random.default_rng(seed)
    perm = perm_rng.permutation(len(sample_ids))
    sample_ids = sample_ids[perm]
    groups = groups.iloc[perm]

    unique_groups = groups.nunique(dropna=False)
    if unique_groups >= 2:
        n_splits = max(2, min(n_splits_desired, unique_groups))
        splitter = GroupKFold(n_splits=n_splits)
        folds = []
        for tr, te in splitter.split(sample_ids, groups=groups):
            folds.append((sample_ids[tr].tolist(), sample_ids[te].tolist()))
        return folds, f"GroupKFold({n_splits})"
    else:
        n_splits = max(2, min(n_splits_desired, len(sample_ids)))
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        folds = []
        for tr, te in splitter.split(sample_ids):
            folds.append((sample_ids[tr].tolist(), sample_ids[te].tolist()))
        return folds, f"KFold({n_splits})"

def preview_outer_splitter(groups: pd.Series, n_obs: int) -> str:
    unique_groups = groups.astype(str).nunique(dropna=False)
    if unique_groups >= 2:
        n_splits = max(2, min(N_SPLITS_DESIRED, unique_groups))
        return f"GroupKFold(n_splits={n_splits})"
    n_splits = max(2, min(N_SPLITS_DESIRED, n_obs))
    return f"KFold(n_splits={n_splits})"

def get_outer_folds_cached(sample_ids: List[str], groups: pd.Series, seed: int, arm_name: str, feature_set: str, compound_id: str):
    split_dir = OUT_CACHE / "splits"
    split_dir.mkdir(parents=True, exist_ok=True)
    cache_payload = {
        "cache_version": CACHE_VERSION,
        "seed": int(seed),
        "arm": arm_name,
        "feature_set": feature_set,
        "compound_id": str(compound_id),
        "sample_ids": list(map(str, sample_ids)),
        "groups": groups.loc[sample_ids].astype(str).to_dict(),
        "n_splits_desired": int(N_SPLITS_DESIRED),
    }
    cache_key = stable_hash(cache_payload, n=24)
    cache_path = split_dir / f"outer_folds__{cache_key}.json"
    if USE_SPLIT_CACHE and cache_path.exists() and not FORCE_RERUN:
        cached = load_json_if_exists(cache_path, default={}) or {}
        folds = [(x[0], x[1]) for x in cached.get("folds", [])]
        splitter_name = cached.get("splitter_name")
        if folds and splitter_name is not None:
            return folds, splitter_name, cache_path

    folds, splitter_name = safe_group_folds(sample_ids, groups, seed, N_SPLITS_DESIRED)
    atomic_write_json({**cache_payload, "splitter_name": splitter_name, "folds": folds}, cache_path)
    return folds, splitter_name, cache_path

def make_train_val_split(train_ids: List[str], groups: pd.Series, seed: int, val_fraction: float = VAL_FRACTION):
    train_ids = np.asarray(train_ids, dtype=object)
    grp = groups.loc[train_ids].astype(str)
    if len(train_ids) < 8:
        n_val = max(1, int(round(len(train_ids) * val_fraction)))
        perm = np.random.default_rng(seed).permutation(len(train_ids))
        val_idx = perm[:n_val]
        tr_idx = perm[n_val:]
        return train_ids[tr_idx].tolist(), train_ids[val_idx].tolist(), "random_small"

    if grp.nunique() >= 2:
        try:
            gss = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
            tr_idx, va_idx = next(gss.split(train_ids, groups=grp))
            return train_ids[tr_idx].tolist(), train_ids[va_idx].tolist(), "GroupShuffleSplit"
        except Exception:
            pass

    tr_ids, va_ids = train_test_split(train_ids, test_size=val_fraction, random_state=seed, shuffle=True)
    return list(tr_ids), list(va_ids), "train_test_split"

def get_train_val_split_cached(train_ids: List[str], groups: pd.Series, seed: int, arm_name: str, feature_set: str, compound_id: str, fold_idx: int):
    split_dir = OUT_CACHE / "splits"
    split_dir.mkdir(parents=True, exist_ok=True)
    cache_payload = {
        "cache_version": CACHE_VERSION,
        "seed": int(seed),
        "arm": arm_name,
        "feature_set": feature_set,
        "compound_id": str(compound_id),
        "fold_idx": int(fold_idx),
        "train_ids": list(map(str, train_ids)),
        "groups": groups.loc[train_ids].astype(str).to_dict(),
        "val_fraction": float(VAL_FRACTION),
    }
    cache_key = stable_hash(cache_payload, n=24)
    cache_path = split_dir / f"train_val_split__{cache_key}.json"
    if USE_SPLIT_CACHE and cache_path.exists() and not FORCE_RERUN:
        cached = load_json_if_exists(cache_path, default={}) or {}
        inner_train_ids = cached.get("inner_train_ids")
        val_ids = cached.get("val_ids")
        splitter_name = cached.get("splitter_name")
        if inner_train_ids and val_ids and splitter_name is not None:
            return inner_train_ids, val_ids, splitter_name, cache_path

    inner_train_ids, val_ids, splitter_name = make_train_val_split(train_ids, groups, seed, VAL_FRACTION)
    atomic_write_json({
        **cache_payload,
        "splitter_name": splitter_name,
        "inner_train_ids": inner_train_ids,
        "val_ids": val_ids,
    }, cache_path)
    return inner_train_ids, val_ids, splitter_name, cache_path

In [11]:

# Fold-safe modality transforms and cached fold features

def select_samples_for_feature_set(arm_name: str, feature_set: str, y_series: pd.Series) -> pd.Series:
    selected_modalities = FEATURE_SET_MODALITIES[feature_set]
    presence = presence_by_arm[arm_name].reindex(index=y_series.index).fillna(0.0)
    keep = presence[list(selected_modalities)].any(axis=1)
    return y_series.loc[keep[keep].index]

def build_pathway_matrix(df: pd.DataFrame, gene_df: pd.DataFrame, selected_gene_sets: Dict[str, set], min_genes: int = 3) -> pd.DataFrame:
    if gene_df.empty or not selected_gene_sets:
        return pd.DataFrame(index=df.index)
    rows = {}
    gene_cols = set(gene_df.columns)
    for name, genes in selected_gene_sets.items():
        use = sorted(gene_cols & genes)
        if len(use) >= min_genes:
            rows[name] = gene_df[use].mean(axis=1)
    if not rows:
        return pd.DataFrame(index=df.index)
    out = pd.DataFrame(rows, index=gene_df.index)
    out.index.name = "depmap_id"
    return to_float32_df(out)

def fit_pca_transform(train_df: pd.DataFrame, modality: str) -> dict:
    train_df = train_df.copy()

    if modality == "mut":
        X_train = train_df.fillna(0.0).to_numpy(dtype=np.float32)
        prevalence = np.asarray((X_train != 0).mean(axis=0)).reshape(-1)
        keep_idx = np.where(prevalence > 0)[0]
        if keep_idx.size == 0:
            return {"mode": "empty", "output_dim": 0}
        if keep_idx.size > PCA_CONFIG["mutation_top_k"]:
            order = np.argsort(prevalence[keep_idx])[::-1][:PCA_CONFIG["mutation_top_k"]]
            keep_idx = keep_idx[order]
        X_train = X_train[:, keep_idx]
        if X_train.shape[1] == 1 or X_train.shape[0] < 3:
            return {
                "mode": "mut_identity",
                "keep_idx": keep_idx.tolist(),
                "output_dim": int(X_train.shape[1]),
            }
        n_comp = int(min(PCA_CONFIG["mutation_svd_components"], X_train.shape[1] - 1, X_train.shape[0] - 1))
        if n_comp < 1:
            return {
                "mode": "mut_identity",
                "keep_idx": keep_idx.tolist(),
                "output_dim": int(X_train.shape[1]),
            }
        svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
        svd.fit(X_train)
        return {
            "mode": "mut_svd",
            "keep_idx": keep_idx.tolist(),
            "svd": svd,
            "output_dim": int(n_comp),
        }

    non_all_nan = train_df.columns[train_df.notna().any(axis=0)].tolist()
    if len(non_all_nan) == 0:
        return {"mode": "empty", "output_dim": 0}

    train_df = train_df[non_all_nan].copy()
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_train = imputer.fit_transform(train_df)
    X_train = scaler.fit_transform(X_train)

    if X_train.shape[0] < PCA_CONFIG["min_train_rows_for_pca"] or X_train.shape[1] == 1:
        return {
            "mode": "continuous_identity",
            "columns": non_all_nan,
            "imputer": imputer,
            "scaler": scaler,
            "output_dim": int(X_train.shape[1]),
        }

    n_comp = int(min(PCA_CONFIG["continuous_components"], X_train.shape[1], X_train.shape[0] - 1))
    if n_comp < 1:
        return {
            "mode": "continuous_identity",
            "columns": non_all_nan,
            "imputer": imputer,
            "scaler": scaler,
            "output_dim": int(X_train.shape[1]),
        }

    pca = PCA(n_components=n_comp, random_state=SEED)
    pca.fit(X_train)
    return {
        "mode": "continuous_pca",
        "columns": non_all_nan,
        "imputer": imputer,
        "scaler": scaler,
        "pca": pca,
        "output_dim": int(n_comp),
    }

def transform_with_fitted(transformer: dict, full_df: pd.DataFrame, modality: str) -> np.ndarray:
    mode = transformer["mode"]
    if mode == "empty":
        return np.zeros((full_df.shape[0], 0), dtype=np.float32)

    if modality == "mut":
        X = full_df.fillna(0.0).to_numpy(dtype=np.float32)
        X = X[:, transformer["keep_idx"]]
        if transformer["mode"] == "mut_identity":
            return X.astype(np.float32)
        return transformer["svd"].transform(X).astype(np.float32)

    X = full_df.reindex(columns=transformer["columns"]).copy()
    X = transformer["imputer"].transform(X)
    X = transformer["scaler"].transform(X)
    if mode == "continuous_identity":
        return X.astype(np.float32)
    return transformer["pca"].transform(X).astype(np.float32)

def build_modality_input_frame(modality: str, arm_name: str) -> pd.DataFrame:
    if FEATURE_REPR_MODE == "pca":
        return modality_frame_for_arm(modality, arm_name)
    if FEATURE_REPR_MODE == "pathway":
        return build_pathway_matrix(
            modality_frame_for_arm(modality, arm_name),
            modality_gene_frame_for_arm(modality, arm_name),
            ALL_GENE_SETS,
        )
    raise KeyError(FEATURE_REPR_MODE)

def sample_missing_fraction(df: pd.DataFrame) -> pd.Series:
    if df.shape[1] == 0:
        return pd.Series(1.0, index=df.index, dtype=np.float32)
    return df.isna().mean(axis=1).astype(np.float32)

def get_fold_feature_bundle_cached(
    arm_name: str,
    feature_set: str,
    compound_id: str,
    seed: int,
    fold_idx: int,
    inner_train_ids: List[str],
    val_ids: List[str],
    test_ids: List[str],
) -> dict:
    cache_dir = OUT_CACHE / "fold_features"
    cache_dir.mkdir(parents=True, exist_ok=True)

    cache_payload = {
        "cache_version": CACHE_VERSION,
        "arm": arm_name,
        "feature_set": feature_set,
        "compound_id": compound_id,
        "seed": int(seed),
        "fold_idx": int(fold_idx),
        "inner_train_ids": list(map(str, inner_train_ids)),
        "val_ids": list(map(str, val_ids)),
        "test_ids": list(map(str, test_ids)),
        "feature_repr_mode": FEATURE_REPR_MODE,
    }
    cache_key = stable_hash(cache_payload, n=24)
    cache_path = cache_dir / f"{cache_key}.npz"
    meta_path = cache_dir / f"{cache_key}.json"

    selected_modalities = FEATURE_SET_MODALITIES[feature_set]
    all_ids = list(map(str, inner_train_ids + val_ids + test_ids))
    splits = {
        "train": inner_train_ids,
        "val": val_ids,
        "test": test_ids,
    }

    if USE_FEATURE_CACHE and cache_path.exists() and meta_path.exists() and not FORCE_RERUN:
        loaded = np.load(cache_path, allow_pickle=False)
        meta = load_json_if_exists(meta_path, default={}) or {}
        bundle = {
            "cache_key": cache_key,
            "modality_dims": meta["modality_dims"],
            "selected_modalities": selected_modalities,
            "splits": {},
        }
        for split_name, split_ids in splits.items():
            bundle["splits"][split_name] = {
                "cell_ids": list(map(str, split_ids)),
                "y": loaded[f"{split_name}__y"].astype(np.float32),
                "presence": loaded[f"{split_name}__presence"].astype(np.float32),
                "missing_frac": loaded[f"{split_name}__missing_frac"].astype(np.float32),
                "inputs": {},
            }
            for modality in selected_modalities:
                bundle["splits"][split_name]["inputs"][modality] = loaded[f"{split_name}__{modality}"].astype(np.float32)
        return bundle

    modality_frames = {
        modality: build_modality_input_frame(modality, arm_name).reindex(index=all_ids)
        for modality in selected_modalities
    }
    modality_transforms = {}
    modality_dims = {}
    for modality in selected_modalities:
        frame = modality_frames[modality]
        tr_frame = frame.reindex(index=inner_train_ids)
        modality_transforms[modality] = fit_pca_transform(tr_frame, modality)
        modality_dims[modality] = int(modality_transforms[modality]["output_dim"])

    y_full = prism_auc[prism_auc["compound_id"] == str(compound_id)].groupby("depmap_id")["y"].mean()

    save_dict = {}
    bundle = {
        "cache_key": cache_key,
        "modality_dims": modality_dims,
        "selected_modalities": selected_modalities,
        "splits": {},
    }

    for split_name, split_ids in splits.items():
        split_inputs = {}
        split_presence_cols = []
        split_missing_cols = []

        for modality in selected_modalities:
            frame = modality_frames[modality].reindex(index=split_ids)
            transformed = transform_with_fitted(modality_transforms[modality], frame, modality)
            split_inputs[modality] = transformed.astype(np.float32)

            base_frame = modality_frame_for_arm(modality, arm_name).reindex(index=split_ids)
            modality_presence = sample_presence_for_frame(base_frame).reindex(split_ids).fillna(0.0).to_numpy(dtype=np.float32)
            modality_missing = sample_missing_fraction(base_frame).reindex(split_ids).fillna(1.0).to_numpy(dtype=np.float32)

            split_presence_cols.append(modality_presence.reshape(-1, 1))
            split_missing_cols.append(modality_missing.reshape(-1, 1))

            save_dict[f"{split_name}__{modality}"] = transformed.astype(np.float32)

        presence_mat = np.concatenate(split_presence_cols, axis=1).astype(np.float32) if split_presence_cols else np.zeros((len(split_ids), 0), dtype=np.float32)
        missing_mat = np.concatenate(split_missing_cols, axis=1).astype(np.float32) if split_missing_cols else np.zeros((len(split_ids), 0), dtype=np.float32)
        y_vec = y_full.reindex(split_ids).to_numpy(dtype=np.float32)

        save_dict[f"{split_name}__presence"] = presence_mat
        save_dict[f"{split_name}__missing_frac"] = missing_mat
        save_dict[f"{split_name}__y"] = y_vec

        bundle["splits"][split_name] = {
            "cell_ids": list(map(str, split_ids)),
            "inputs": split_inputs,
            "presence": presence_mat,
            "missing_frac": missing_mat,
            "y": y_vec,
        }

    np.savez_compressed(cache_path, **save_dict)
    write_cache_metadata(meta_path, {
        **cache_payload,
        "modality_dims": modality_dims,
    })
    return bundle

In [12]:

# Dataset and gated cross-modal attention model

class FusionDataset(Dataset):
    def __init__(self, cell_ids: List[str], modality_inputs: Dict[str, np.ndarray], presence: np.ndarray, missing_frac: np.ndarray, y: np.ndarray):
        self.cell_ids = list(map(str, cell_ids))
        self.modality_inputs = {k: np.asarray(v, dtype=np.float32) for k, v in modality_inputs.items()}
        self.presence = np.asarray(presence, dtype=np.float32)
        self.missing_frac = np.asarray(missing_frac, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.float32)

    def __len__(self) -> int:
        return len(self.cell_ids)

    def __getitem__(self, idx: int) -> dict:
        return {
            "cell_id": self.cell_ids[idx],
            "inputs": {k: torch.from_numpy(v[idx]) for k, v in self.modality_inputs.items()},
            "presence": torch.from_numpy(self.presence[idx]),
            "missing_frac": torch.from_numpy(self.missing_frac[idx]),
            "y": torch.tensor(self.y[idx], dtype=torch.float32),
        }

def collate_fusion_batch(batch: List[dict]) -> dict:
    keys = list(batch[0]["inputs"].keys())
    return {
        "cell_id": [x["cell_id"] for x in batch],
        "inputs": {k: torch.stack([x["inputs"][k] for x in batch], dim=0) for k in keys},
        "presence": torch.stack([x["presence"] for x in batch], dim=0),
        "missing_frac": torch.stack([x["missing_frac"] for x in batch], dim=0),
        "y": torch.stack([x["y"] for x in batch], dim=0),
    }

class ModalityTower(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, out_dim: int, dropout: float):
        super().__init__()
        if input_dim <= 0:
            raise ValueError("ModalityTower input_dim must be positive.")
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class GatedCrossModalAttentionRegressor(nn.Module):
    def __init__(
        self,
        modality_dims: Dict[str, int],
        modality_order: Tuple[str, ...],
        tower_hidden_dim: int = 256,
        tower_out_dim: int = 128,
        fusion_heads: int = 4,
        dropout: float = 0.2,
        modality_dropout_p: float = 0.0,
    ):
        super().__init__()
        self.modality_order = tuple(modality_order)
        self.modality_dropout_p = float(modality_dropout_p)

        self.towers = nn.ModuleDict({
            modality: ModalityTower(
                input_dim=int(modality_dims[modality]),
                hidden_dim=tower_hidden_dim,
                out_dim=tower_out_dim,
                dropout=dropout,
            )
            for modality in self.modality_order
        })

        self.self_attn = nn.MultiheadAttention(
            embed_dim=tower_out_dim,
            num_heads=fusion_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attn_norm = nn.LayerNorm(tower_out_dim)
        self.token_dropout = nn.Dropout(dropout)

        gate_in_dim = tower_out_dim + len(self.modality_order) * 2
        self.gate_mlp = nn.Sequential(
            nn.Linear(gate_in_dim, tower_out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(tower_out_dim, len(self.modality_order)),
        )

        head_in_dim = tower_out_dim + len(self.modality_order) * 2
        self.head = nn.Sequential(
            nn.Linear(head_in_dim, tower_out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(tower_out_dim, 1),
        )

    def _apply_modality_dropout(self, presence: torch.Tensor) -> torch.Tensor:
        if (not self.training) or self.modality_dropout_p <= 0.0 or presence.shape[1] <= 1:
            return presence

        keep = presence.clone()
        bern = torch.rand_like(keep)
        drop_mask = (bern < self.modality_dropout_p) & (presence > 0.5)
        keep = torch.where(drop_mask, torch.zeros_like(keep), keep)

        zero_rows = keep.sum(dim=1) < 0.5
        if zero_rows.any():
            fallback_idx = presence[zero_rows].argmax(dim=1)
            keep[zero_rows] = 0.0
            keep[zero_rows, fallback_idx] = 1.0
        return keep

    def forward(self, batch: dict, return_aux: bool = False):
        presence = batch["presence"]
        missing_frac = batch["missing_frac"]
        effective_presence = self._apply_modality_dropout(presence)

        token_list = []
        for i, modality in enumerate(self.modality_order):
            x = batch["inputs"][modality]
            tok = self.towers[modality](x)
            tok = tok * effective_presence[:, i:i+1]
            token_list.append(tok)

        tokens = torch.stack(token_list, dim=1)
        key_padding_mask = effective_presence <= 0.5

        attn_out, attn_weights = self.self_attn(
            tokens, tokens, tokens,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        tokens = self.attn_norm(tokens + self.token_dropout(attn_out))

        pooled = tokens.mean(dim=1)
        gate_context = torch.cat([pooled, effective_presence, missing_frac], dim=1)
        gate_logits = self.gate_mlp(gate_context)
        gate_logits = gate_logits.masked_fill(effective_presence <= 0.5, -1e9)
        gate_weights = torch.softmax(gate_logits, dim=1)

        fused = torch.sum(tokens * gate_weights.unsqueeze(-1), dim=1)
        head_input = torch.cat([fused, effective_presence, missing_frac], dim=1)
        pred = self.head(head_input).view(-1)

        if not return_aux:
            return pred

        attn_mean = attn_weights.mean(dim=1)
        return pred, {
            "gate_weights": gate_weights,
            "effective_presence": effective_presence,
            "attention_mean": attn_mean,
        }

In [13]:

# Training and evaluation helpers

def make_loss(name: str):
    if name == "smooth_l1":
        return nn.SmoothL1Loss()
    if name == "mse":
        return nn.MSELoss()
    raise KeyError(name)

def batch_to_device(batch: dict, device: torch.device) -> dict:
    return {
        "cell_id": batch["cell_id"],
        "inputs": {k: v.to(device) for k, v in batch["inputs"].items()},
        "presence": batch["presence"].to(device),
        "missing_frac": batch["missing_frac"].to(device),
        "y": batch["y"].to(device),
    }

def train_one_epoch(model, loader, optimiser, criterion, device):
    model.train()
    losses = []
    for batch in loader:
        batch = batch_to_device(batch, device)
        optimiser.zero_grad(set_to_none=True)
        pred = model(batch)
        loss = criterion(pred, batch["y"].view(-1))
        loss.backward()
        if MODEL_CONFIG["grad_clip_norm"] is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MODEL_CONFIG["grad_clip_norm"])
        optimiser.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else np.nan

@torch.no_grad()
def predict_loader(model, loader, device, return_aux: bool = True):
    model.eval()
    preds = []
    ys = []
    cell_ids = []
    gate_rows = []
    for batch in loader:
        batch = batch_to_device(batch, device)
        if return_aux:
            pred, aux = model(batch, return_aux=True)
            gate = aux["gate_weights"].detach().cpu().numpy()
            eff = aux["effective_presence"].detach().cpu().numpy()
            attn = aux["attention_mean"].detach().cpu().numpy()
        else:
            pred = model(batch)
            gate = None
            eff = None
            attn = None

        preds.append(pred.detach().cpu().numpy())
        ys.append(batch["y"].view(-1).detach().cpu().numpy())
        cell_ids.extend(batch["cell_id"])

        if gate is not None:
            for i, cid in enumerate(batch["cell_id"]):
                row = {
                    "depmap_id": cid,
                    "y_true": float(batch["y"][i].detach().cpu()),
                    "y_pred": float(pred[i].detach().cpu()),
                }
                for j, modality in enumerate(model.modality_order):
                    row[f"gate_{modality}"] = float(gate[i, j])
                    row[f"present_{modality}"] = float(batch["presence"][i, j].detach().cpu())
                    row[f"effective_present_{modality}"] = float(eff[i, j])
                    row[f"missing_frac_{modality}"] = float(batch["missing_frac"][i, j].detach().cpu())
                    row[f"attn_self_{modality}"] = float(attn[i, j, j])
                gate_rows.append(row)

    pred_arr = np.concatenate(preds) if preds else np.array([])
    y_arr = np.concatenate(ys) if ys else np.array([])
    gate_df = pd.DataFrame(gate_rows)
    return pred_arr, y_arr, cell_ids, gate_df

def fit_gated_attention_model(
    train_dataset: FusionDataset,
    val_dataset: FusionDataset,
    modality_dims: Dict[str, int],
    modality_order: Tuple[str, ...],
    seed: int,
    run_key: str,
    batch_size: Optional[int] = None,
):
    set_all_seeds(seed)

    model_dir = OUT_CACHE / "models"
    history_dir = OUT_CACHE / "histories"
    train_state_dir = OUT_CACHE / "train_states"
    fit_meta_dir = OUT_CACHE / "fit_meta"
    for d in [model_dir, history_dir, train_state_dir, fit_meta_dir]:
        d.mkdir(parents=True, exist_ok=True)

    model_path = model_dir / f"{run_key}.pt"
    history_path = history_dir / f"{run_key}.csv"
    train_state_path = train_state_dir / f"{run_key}.pt"
    fit_meta_path = fit_meta_dir / f"{run_key}.json"

    model = GatedCrossModalAttentionRegressor(
        modality_dims=modality_dims,
        modality_order=modality_order,
        tower_hidden_dim=MODEL_CONFIG["tower_hidden_dim"],
        tower_out_dim=MODEL_CONFIG["tower_out_dim"],
        fusion_heads=MODEL_CONFIG["fusion_heads"],
        dropout=MODEL_CONFIG["dropout"],
        modality_dropout_p=MODEL_CONFIG["track2_modality_dropout"],
    ).to(DEVICE)

    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=MODEL_CONFIG["lr"],
        weight_decay=MODEL_CONFIG["weight_decay"],
    )
    criterion = make_loss(MODEL_CONFIG["loss"])

    effective_batch_size = int(batch_size or MODEL_CONFIG["batch_size"])

    train_loader = DataLoader(
        train_dataset,
        batch_size=effective_batch_size,
        shuffle=True,
        num_workers=MODEL_CONFIG["num_workers"],
        collate_fn=collate_fusion_batch,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=effective_batch_size,
        shuffle=False,
        num_workers=MODEL_CONFIG["num_workers"],
        collate_fn=collate_fusion_batch,
    )

    if model_path.exists() and history_path.exists() and fit_meta_path.exists() and not FORCE_RERUN:
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        history_df = pd.read_csv(history_path)
        fit_info = load_json_if_exists(fit_meta_path, default={}) or {}
        fit_info.update({
            "resumed_from_epoch_checkpoint": False,
            "model_path": str(model_path),
            "history_path": str(history_path),
            "train_state_path": str(train_state_path),
            "fit_meta_path": str(fit_meta_path),
        })
        return model, history_df, fit_info

    best_state = None
    best_epoch = -1
    best_val_spearman = -np.inf
    patience_counter = 0
    history = []
    start_epoch = 1
    resumed_from_epoch_checkpoint = False

    if USE_MODEL_STATE_RESUME and train_state_path.exists() and not FORCE_RERUN:
        ckpt = torch.load(train_state_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        optimiser.load_state_dict(ckpt["optimizer_state_dict"])
        best_state = ckpt.get("best_state_dict")
        best_epoch = int(ckpt.get("best_epoch", -1))
        best_val_spearman = float(ckpt.get("best_val_spearman", -np.inf))
        patience_counter = int(ckpt.get("patience_counter", 0))
        history = list(ckpt.get("history", []))
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        resumed_from_epoch_checkpoint = True

    for epoch in range(start_epoch, MODEL_CONFIG["max_epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimiser, criterion, DEVICE)
        val_pred, val_y, _, _ = predict_loader(model, val_loader, DEVICE, return_aux=False)
        val_s = spearman_safe(val_y, val_pred)
        val_r2 = r2_safe(val_y, val_pred)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_spearman": val_s,
            "val_r2": val_r2,
        })

        monitor = -np.inf if np.isnan(val_s) else val_s
        if monitor > best_val_spearman + MODEL_CONFIG["min_delta"]:
            best_val_spearman = monitor
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if TRAIN_STATE_SAVE_EVERY_EPOCHS and (epoch % TRAIN_STATE_SAVE_EVERY_EPOCHS == 0):
            torch.save({
                "cache_version": CACHE_VERSION,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimiser.state_dict(),
                "best_state_dict": best_state,
                "best_epoch": best_epoch,
                "best_val_spearman": best_val_spearman,
                "patience_counter": patience_counter,
                "history": history,
                "seed": int(seed),
                "run_key": run_key,
            }, train_state_path)

        if patience_counter >= MODEL_CONFIG["patience"]:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)
    fit_info = {
        "best_epoch": int(best_epoch),
        "best_val_spearman": None if best_val_spearman == -np.inf else float(best_val_spearman),
        "n_epochs_run": int(len(history)),
        "resumed_from_epoch_checkpoint": bool(resumed_from_epoch_checkpoint),
        "model_path": str(model_path),
        "history_path": str(history_path),
        "train_state_path": str(train_state_path),
        "fit_meta_path": str(fit_meta_path),
    }

    atomic_to_csv(history_df, history_path, index=False)
    torch.save(model.state_dict(), model_path)
    write_cache_metadata(fit_meta_path, {
        "cache_version": CACHE_VERSION,
        **fit_info,
        "seed": int(seed),
        "run_key": run_key,
    })
    return model, history_df, fit_info

In [14]:
# Benchmark loop, per-mode execution, and final PCA vs pathway comparison
def normalise_feature_set_name(fs: str) -> str:
    return "+".join(str(fs).split("+"))

def discover_tabular_baselines(scan_dirs: List[Path]) -> pd.DataFrame:
    records = []
    for d in scan_dirs:
        if not d.exists():
            continue
        for path in list(d.rglob("*.csv")) + list(d.rglob("*.parquet")):
            try:
                df = read_table(path)
            except Exception:
                continue

            cols_lower = {str(c).lower(): c for c in df.columns}
            colmap = {}
            for target_name, cands in {
                "arm": ["arm", "proteomics_arm"],
                "feature_set": ["feature_set", "features", "feature_combo"],
                "compound_id": ["compound_id", "drug_id", "compound"],
                "mean_spearman": ["mean_spearman", "spearman_mean", "cv_mean_spearman", "spearman"],
            }.items():
                for cand in cands:
                    if cand in cols_lower:
                        colmap[cols_lower[cand]] = target_name
                        break

            if set(colmap.values()) >= {"arm", "feature_set", "mean_spearman"}:
                tmp = df.rename(columns=colmap).copy()
                tmp["source_file"] = str(path)
                if "compound_id" not in tmp.columns:
                    tmp["compound_id"] = "__all__"
                tmp = tmp[["arm", "feature_set", "compound_id", "mean_spearman", "source_file"]].copy()
                records.append(tmp)

    if not records:
        return pd.DataFrame(columns=["arm", "feature_set", "compound_id", "mean_spearman", "source_file"])

    out = pd.concat(records, ignore_index=True)
    out["arm"] = out["arm"].astype(str)
    out["feature_set"] = out["feature_set"].astype(str).map(normalise_feature_set_name)
    out["compound_id"] = out["compound_id"].astype(str)
    out["mean_spearman"] = pd.to_numeric(out["mean_spearman"], errors="coerce")
    out = out.dropna(subset=["mean_spearman"])
    return out

def run_single_mode(mode: str) -> dict:
    set_run_mode(mode)

    run_metadata = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "project_root": str(PROJECT_ROOT),
        "device": str(DEVICE),
        "seeds": SEEDS,
        "target": PRIMARY_TARGET,
        "top_k_drugs": TOP_K_DRUGS,
        "n_splits_desired": N_SPLITS_DESIRED,
        "min_cells_per_drug": MIN_CELLS_PER_DRUG,
        "feature_sets": FEATURE_SETS,
        "feature_repr_mode": FEATURE_REPR_MODE,
        "arm_thresholds": ARM_THRESHOLDS,
        "pca_config": PCA_CONFIG,
        "model_config": MODEL_CONFIG,
        "gmt_path": None if GMT_PATH is None else str(GMT_PATH),
    }
    atomic_write_json(run_metadata, OUT_META / "run_metadata.json")
    atomic_write_json({"top10_drugs": TOP_DRUGS}, OUT_META / "top10_drugs.json")

    checkpoint_path = OUT_RESULTS / "gated_attention_checkpoint.parquet"
    run_state_path = OUT_META / "run_state.json"
    per_fold_rows = []

    if checkpoint_path.exists() and RESUME_FROM_CHECKPOINT and not FORCE_RERUN:
        try:
            prev = pd.read_parquet(checkpoint_path)
            per_fold_rows = prev.to_dict("records")
            print(f"Loaded existing checkpoint rows for mode={mode}: {len(per_fold_rows)}")
        except Exception:
            print(f"Could not load prior checkpoint for mode={mode}; starting fresh.")

    completed_keys = {
        (
            r["seed"],
            r["arm"],
            r["feature_set"],
            r["compound_id"],
            r["fold_idx"],
        )
        for r in per_fold_rows
    } if per_fold_rows else set()

    def save_checkpoint(rows: List[dict], last_completed: Optional[dict] = None):
        if not rows:
            return
        df = pd.DataFrame(rows)
        atomic_to_parquet(df, checkpoint_path)
        update_run_state(run_state_path, {
            "cache_version": CACHE_VERSION,
            "resume_from_checkpoint": bool(RESUME_FROM_CHECKPOINT),
            "force_rerun": bool(FORCE_RERUN),
            "n_completed_rows": int(df.shape[0]),
            "last_completed": last_completed,
            "checkpoint_path": str(checkpoint_path),
            "feature_repr_mode": FEATURE_REPR_MODE,
        })

    run_arms = list(ARM_THRESHOLDS.keys())
    run_features = list(FEATURE_SETS)
    run_drugs = list(TOP_DRUGS)

    if FAST_DEV_RUN:
        if FAST_DEV_LIMIT_ARMS is not None:
            run_arms = run_arms[:FAST_DEV_LIMIT_ARMS]
        if FAST_DEV_LIMIT_FEATURES is not None:
            run_features = run_features[:FAST_DEV_LIMIT_FEATURES]
        if FAST_DEV_LIMIT_DRUGS is not None:
            run_drugs = run_drugs[:FAST_DEV_LIMIT_DRUGS]

    rows_since_last_checkpoint = 0

    for seed in SEEDS:
        set_all_seeds(seed)
        print(f"\nMode={mode} | Seed {seed}")

        for arm_name in run_arms:
            eligible_cells = list(prot_arm_data[arm_name]["eligible_cells"])
            if len(eligible_cells) == 0:
                print(f"    {arm_name}: eligible_cells=0, skipping arm")
                continue

            print(f"    Arm: {arm_name} | eligible_cells={len(eligible_cells)}")
            arm_base_groups = cell_index.loc[eligible_cells, group_col].astype(str)

            for feature_set in run_features:
                selected_modalities = FEATURE_SET_MODALITIES[feature_set]
                last_completed_feature = None
                feature_rows_before = len(per_fold_rows)

                for drug_idx, compound_id in enumerate(run_drugs, start=1):
                    y_series = prism_auc[prism_auc["compound_id"] == str(compound_id)].groupby("depmap_id")["y"].mean()
                    y_series = y_series.reindex(eligible_cells).dropna()
                    y_series = select_samples_for_feature_set(arm_name, feature_set, y_series)

                    if y_series.shape[0] < MIN_CELLS_PER_DRUG:
                        continue

                    sample_ids = y_series.index.tolist()
                    sample_groups = arm_base_groups.loc[sample_ids]
                    splitter_preview = preview_outer_splitter(sample_groups, len(sample_ids))

                    if drug_idx == 1 or drug_idx % 5 == 0 or drug_idx == len(run_drugs):
                        print(
                            f"      [{feature_set}] drug {drug_idx}/{len(run_drugs)} | "
                            f"{compound_id} | n_cells={len(sample_ids)} | {splitter_preview}",
                            flush=True,
                        )

                    outer_folds, outer_splitter, outer_split_cache = get_outer_folds_cached(
                        sample_ids, sample_groups, seed, arm_name, feature_set, compound_id
                    )

                    for fold_idx, (train_ids, test_ids) in enumerate(outer_folds):
                        completion_key = (seed, arm_name, feature_set, compound_id, fold_idx)
                        if completion_key in completed_keys:
                            continue

                        inner_train_ids, val_ids, val_splitter, inner_split_cache = get_train_val_split_cached(
                            train_ids=train_ids,
                            groups=sample_groups,
                            seed=seed + fold_idx,
                            arm_name=arm_name,
                            feature_set=feature_set,
                            compound_id=compound_id,
                            fold_idx=fold_idx,
                        )

                        feature_bundle = get_fold_feature_bundle_cached(
                            arm_name=arm_name,
                            feature_set=feature_set,
                            compound_id=compound_id,
                            seed=seed,
                            fold_idx=fold_idx,
                            inner_train_ids=inner_train_ids,
                            val_ids=val_ids,
                            test_ids=test_ids,
                        )

                        modality_dims = feature_bundle["modality_dims"]
                        if any(int(modality_dims[m]) <= 0 for m in selected_modalities):
                            continue

                        train_pack = feature_bundle["splits"]["train"]
                        val_pack = feature_bundle["splits"]["val"]
                        test_pack = feature_bundle["splits"]["test"]

                        train_ds = FusionDataset(
                            train_pack["cell_ids"], train_pack["inputs"], train_pack["presence"], train_pack["missing_frac"], train_pack["y"]
                        )
                        val_ds = FusionDataset(
                            val_pack["cell_ids"], val_pack["inputs"], val_pack["presence"], val_pack["missing_frac"], val_pack["y"]
                        )
                        test_ds = FusionDataset(
                            test_pack["cell_ids"], test_pack["inputs"], test_pack["presence"], test_pack["missing_frac"], test_pack["y"]
                        )

                        run_key = "__".join([
                            f"mode{safe_slug(mode)}",
                            f"seed{seed}",
                            safe_slug(arm_name),
                            safe_slug(feature_set.replace('+', '_')),
                            safe_slug(compound_id),
                            f"fold{fold_idx}",
                        ])

                        model, history_df, fit_info = fit_gated_attention_model(
                            train_dataset=train_ds,
                            val_dataset=val_ds,
                            modality_dims=modality_dims,
                            modality_order=selected_modalities,
                            seed=seed + fold_idx,
                            run_key=run_key,
                            batch_size=get_batch_size_for_arm(arm_name),
                        )

                        test_loader = DataLoader(
                            test_ds,
                            batch_size=get_batch_size_for_arm(arm_name),
                            shuffle=False,
                            num_workers=0,
                            collate_fn=collate_fusion_batch,
                        )
                        test_pred, test_y_out, test_cell_ids, gate_df = predict_loader(model, test_loader, DEVICE, return_aux=True)

                        pred_df = pd.DataFrame({
                            "depmap_id": test_cell_ids,
                            "y_true": test_y_out,
                            "y_pred": test_pred,
                            "seed": seed,
                            "mode": mode,
                            "arm": arm_name,
                            "feature_set": feature_set,
                            "compound_id": compound_id,
                            "fold_idx": fold_idx,
                        })
                        pred_path = OUT_RESULTS / f"predictions__{run_key}.csv"
                        atomic_to_csv(pred_df, pred_path, index=False)

                        if not gate_df.empty:
                            gate_df["seed"] = seed
                            gate_df["mode"] = mode
                            gate_df["arm"] = arm_name
                            gate_df["feature_set"] = feature_set
                            gate_df["compound_id"] = compound_id
                            gate_df["fold_idx"] = fold_idx
                        gate_path = OUT_INTERP / f"gate_weights__{run_key}.csv"
                        atomic_to_csv(gate_df, gate_path, index=False)

                        row = {
                            "seed": seed,
                            "mode": mode,
                            "arm": arm_name,
                            "feature_set": feature_set,
                            "compound_id": compound_id,
                            "fold_idx": fold_idx,
                            "run_key": run_key,
                            "n_train": len(inner_train_ids),
                            "n_val": len(val_ids),
                            "n_test": len(test_ids),
                            "outer_splitter": outer_splitter,
                            "val_splitter": val_splitter,
                            "spearman": spearman_safe(test_y_out, test_pred),
                            "r2": r2_safe(test_y_out, test_pred),
                            "best_epoch": fit_info["best_epoch"],
                            "best_val_spearman": fit_info["best_val_spearman"],
                            "n_epochs_run": fit_info["n_epochs_run"],
                            "resumed_from_epoch_checkpoint": fit_info.get("resumed_from_epoch_checkpoint", False),
                            "outer_split_cache": str(outer_split_cache),
                            "inner_split_cache": str(inner_split_cache),
                            "feature_cache_key": feature_bundle["cache_key"],
                            "predictions_path": str(pred_path),
                            "gate_weights_path": str(gate_path),
                            "model_path": fit_info.get("model_path"),
                            "history_path": fit_info.get("history_path"),
                            "train_state_path": fit_info.get("train_state_path"),
                            "fit_meta_path": fit_info.get("fit_meta_path"),
                        }

                        for modality in ALL_MODALITIES:
                            if modality in selected_modalities and not gate_df.empty:
                                row[f"mean_gate_{modality}"] = float(gate_df[f"gate_{modality}"].mean())
                                row[f"present_frac_{modality}"] = float(gate_df[f"present_{modality}"].mean())
                                present_mask = gate_df[f"present_{modality}"] > 0.5
                                if present_mask.sum() >= 3:
                                    row[f"gate_missing_spearman_{modality}"] = spearman_safe(
                                        gate_df.loc[present_mask, f"missing_frac_{modality}"],
                                        gate_df.loc[present_mask, f"gate_{modality}"],
                                    )
                                else:
                                    row[f"gate_missing_spearman_{modality}"] = np.nan
                            else:
                                row[f"mean_gate_{modality}"] = np.nan
                                row[f"present_frac_{modality}"] = np.nan
                                row[f"gate_missing_spearman_{modality}"] = np.nan

                        per_fold_rows.append(row)
                        completed_keys.add(completion_key)
                        rows_since_last_checkpoint += 1
                        last_completed_feature = {
                            "mode": mode,
                            "seed": seed,
                            "arm": arm_name,
                            "feature_set": feature_set,
                            "compound_id": compound_id,
                            "fold_idx": fold_idx,
                        }

                        print(
                            f"        finished mode={mode} | {arm_name} | {feature_set} | "
                            f"drug {drug_idx}/{len(run_drugs)} | fold {fold_idx + 1}/{len(outer_folds)}",
                            flush=True,
                        )

                        del model, train_ds, val_ds, test_ds, test_loader
                        gc.collect()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

                        if CHECKPOINT_EVERY_N_ROWS and rows_since_last_checkpoint >= CHECKPOINT_EVERY_N_ROWS:
                            save_checkpoint(per_fold_rows, last_completed=last_completed_feature)
                            rows_since_last_checkpoint = 0

                if CHECKPOINT_EVERY_N_DRUGS and (
                    drug_idx % CHECKPOINT_EVERY_N_DRUGS == 0 or drug_idx == len(run_drugs)
                ):
                    save_checkpoint(per_fold_rows, last_completed=last_completed_feature)
                    print(f"        [mid-checkpoint] drug {drug_idx}/{len(run_drugs)}", flush=True)

            feature_rows_after = len(per_fold_rows)
            print(
                f"      completed feature_set={feature_set} | new_rows={feature_rows_after - feature_rows_before}",
                flush=True,
            )

    save_checkpoint(per_fold_rows)
    print(f"Benchmark loop complete for mode={mode}. Rows: {len(per_fold_rows)}")

    if not per_fold_rows and checkpoint_path.exists():
        per_fold_df = pd.read_parquet(checkpoint_path)
    else:
        per_fold_df = pd.DataFrame(per_fold_rows)

    if per_fold_df.empty:
        raise RuntimeError(f"No benchmark rows were produced for mode={mode}. Check cohort sizes, thresholds, and minimum-cell settings.")

    per_fold_df = per_fold_df.sort_values(
        ["seed", "arm", "feature_set", "compound_id", "fold_idx"]
    ).reset_index(drop=True)

    per_fold_df.to_csv(OUT_RESULTS / "per_fold_results.csv", index=False)
    per_fold_df.to_parquet(OUT_RESULTS / "per_fold_results.parquet", index=False)

    per_drug_seed = (
        per_fold_df.groupby(["seed", "arm", "feature_set", "compound_id"], as_index=False)
        .agg(
            mean_spearman=("spearman", "mean"),
            median_spearman=("spearman", "median"),
            mean_r2=("r2", "mean"),
            median_r2=("r2", "median"),
            n_folds=("fold_idx", "nunique"),
        )
    )
    per_drug_seed.to_csv(OUT_RESULTS / "per_drug_seed_summary.csv", index=False)

    per_drug = (
        per_drug_seed.groupby(["arm", "feature_set", "compound_id"], as_index=False)
        .agg(
            mean_spearman=("mean_spearman", "mean"),
            median_spearman=("median_spearman", "median"),
            mean_r2=("mean_r2", "mean"),
            median_r2=("median_r2", "median"),
            n_seed_rows=("seed", "nunique"),
        )
    )
    per_drug.to_csv(OUT_RESULTS / "per_drug_summary.csv", index=False)

    seed_summary = (
        per_fold_df.groupby(["seed"], as_index=False)
        .agg(
            mean_spearman=("spearman", "mean"),
            median_spearman=("spearman", "median"),
            mean_r2=("r2", "mean"),
            median_r2=("r2", "median"),
            n_rows=("fold_idx", "count"),
        )
    )
    seed_summary.to_csv(OUT_RESULTS / "seed_summary.csv", index=False)

    arm_feature_summary = (
        per_drug_seed.groupby(["arm", "feature_set"], as_index=False)
        .agg(
            mean_spearman=("mean_spearman", "mean"),
            median_spearman=("median_spearman", "median"),
            mean_r2=("mean_r2", "mean"),
            median_r2=("median_r2", "median"),
            n_drug_seed=("compound_id", "count"),
        )
        .sort_values(["arm", "mean_spearman"], ascending=[True, False])
    )
    arm_feature_summary.to_csv(OUT_RESULTS / "arm_feature_summary.csv", index=False)

    arm_summary = (
        per_drug_seed.groupby(["arm"], as_index=False)
        .agg(
            mean_spearman=("mean_spearman", "mean"),
            median_spearman=("median_spearman", "median"),
            mean_r2=("mean_r2", "mean"),
            median_r2=("median_r2", "median"),
            n_rows=("compound_id", "count"),
        )
        .sort_values(["mean_spearman"], ascending=[False])
    )
    arm_summary.to_csv(OUT_RESULTS / "arm_summary.csv", index=False)

    best_by_arm = (
        arm_feature_summary
        .sort_values(["arm", "mean_spearman"], ascending=[True, False])
        .groupby(["arm"], as_index=False)
        .first()
    )
    best_by_arm.to_csv(OUT_RESULTS / "best_feature_set_by_arm.csv", index=False)

    display(Markdown(f"## Mode `{mode}` summaries"))
    display(seed_summary)
    display(arm_summary)
    display(best_by_arm)

    gate_paths = sorted(OUT_INTERP.glob("gate_weights__*.csv"))
    gate_summary_rows = []

    for path in gate_paths:
        try:
            gdf = pd.read_csv(path)
        except Exception:
            continue
        if gdf.empty:
            continue

        meta_cols = ["seed", "arm", "feature_set", "compound_id", "fold_idx"]
        meta = {c: gdf[c].iloc[0] for c in meta_cols if c in gdf.columns}

        for modality in ALL_MODALITIES:
            gate_col = f"gate_{modality}"
            present_col = f"present_{modality}"
            miss_col = f"missing_frac_{modality}"
            if gate_col not in gdf.columns:
                continue
            present_mask = gdf[present_col] > 0.5 if present_col in gdf.columns else pd.Series(False, index=gdf.index)
            corr = spearman_safe(
                gdf.loc[present_mask, miss_col],
                gdf.loc[present_mask, gate_col]
            ) if (present_mask.sum() >= 3 and miss_col in gdf.columns) else np.nan

            gate_summary_rows.append({
                **meta,
                "mode": mode,
                "modality": modality,
                "mean_gate": float(gdf[gate_col].mean()),
                "mean_gate_present_only": float(gdf.loc[present_mask, gate_col].mean()) if present_mask.any() else np.nan,
                "present_fraction": float(gdf[present_col].mean()) if present_col in gdf.columns else np.nan,
                "gate_missing_spearman": corr,
                "source_file": str(path),
            })

    gate_summary_df = pd.DataFrame(gate_summary_rows)
    gate_summary_df.to_csv(OUT_INTERP / "gate_summary_per_fold.csv", index=False)

    gate_summary_agg = (
        gate_summary_df.groupby(["arm", "feature_set", "modality"], as_index=False)
        .agg(
            mean_gate=("mean_gate", "mean"),
            mean_gate_present_only=("mean_gate_present_only", "mean"),
            mean_present_fraction=("present_fraction", "mean"),
            mean_gate_missing_spearman=("gate_missing_spearman", "mean"),
            n_rows=("compound_id", "count"),
        )
        .sort_values(["arm", "feature_set", "modality"])
    )
    gate_summary_agg.to_csv(OUT_INTERP / "gate_summary_aggregated.csv", index=False)

    display(gate_summary_agg.head(20))

    baseline_df = discover_tabular_baselines(BASELINE_SCAN_DIRS)
    baseline_df.to_csv(OUT_COMPARE / "discovered_tabular_baselines.csv", index=False)

    graph_compare = arm_feature_summary.copy()
    graph_compare["compound_id"] = "__all__"

    baseline_compare = graph_compare.merge(
        baseline_df,
        on=["arm", "feature_set", "compound_id"],
        how="left",
        suffixes=("_gated_attention", "_baseline"),
    )
    if not baseline_compare.empty and "mean_spearman_baseline" in baseline_compare.columns:
        baseline_compare["delta_vs_baseline"] = (
            baseline_compare["mean_spearman_gated_attention"] - baseline_compare["mean_spearman_baseline"]
        )

    baseline_compare.to_csv(OUT_COMPARE / "arm_feature_vs_baseline.csv", index=False)
    display(baseline_compare.head(20))

    plt.figure(figsize=(6, 4))
    plt.bar(seed_summary["seed"].astype(str), seed_summary["mean_spearman"])
    plt.title(f"{mode}: Mean Spearman by seed")
    plt.xlabel("Seed")
    plt.ylabel("Mean Spearman")
    plt.tight_layout()
    plt.savefig(OUT_FIGS / "seed_mean_spearman.png", dpi=180, bbox_inches="tight")
    plt.show()

    for arm_name, sub in arm_feature_summary.groupby("arm"):
        ordered = sub.sort_values("mean_spearman", ascending=False).reset_index(drop=True)
        plt.figure(figsize=(8, max(4, 0.28 * len(ordered))))
        plt.barh(ordered["feature_set"], ordered["mean_spearman"])
        plt.gca().invert_yaxis()
        plt.title(f"{mode} | {arm_name}: mean Spearman by feature set")
        plt.xlabel("Mean Spearman")
        plt.tight_layout()
        plt.savefig(
            OUT_FIGS / f"{safe_slug(arm_name)}__feature_set_bar.png",
            dpi=180,
            bbox_inches="tight",
        )
        plt.show()

    top_configs = per_drug.sort_values("mean_spearman", ascending=False).head(20)
    plt.figure(figsize=(9, max(4, 0.30 * len(top_configs))))
    labels = [f"{r.arm} | {r.feature_set} | {r.compound_id}" for r in top_configs.itertuples()]
    plt.barh(labels, top_configs["mean_spearman"])
    plt.gca().invert_yaxis()
    plt.title(f"{mode}: Top 20 gated cross-modal attention drug-level configurations")
    plt.xlabel("Mean Spearman")
    plt.tight_layout()
    plt.savefig(OUT_FIGS / "top20_drug_level_configurations.png", dpi=180, bbox_inches="tight")
    plt.show()

    if not gate_summary_agg.empty:
        for arm_name, sub in gate_summary_agg.groupby("arm"):
            pivot = sub.groupby("modality", as_index=False)["mean_gate_present_only"].mean()
            plt.figure(figsize=(6, 4))
            plt.bar(pivot["modality"], pivot["mean_gate_present_only"])
            plt.title(f"{mode} | {arm_name}: mean gate when modality is present")
            plt.xlabel("Modality")
            plt.ylabel("Mean gate")
            plt.tight_layout()
            plt.savefig(
                OUT_FIGS / f"{safe_slug(arm_name)}__mean_gate_present.png",
                dpi=180,
                bbox_inches="tight",
            )
            plt.show()

    final_lock = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target": PRIMARY_TARGET,
        "top_drugs": TOP_DRUGS,
        "seeds": SEEDS,
        "n_splits_desired": N_SPLITS_DESIRED,
        "min_cells_per_drug": MIN_CELLS_PER_DRUG,
        "group_column": group_col,
        "feature_repr_mode": FEATURE_REPR_MODE,
        "arm_thresholds": ARM_THRESHOLDS,
        "feature_sets": FEATURE_SETS,
        "pca_config": PCA_CONFIG,
        "model_config": MODEL_CONFIG,
        "best_feature_set_by_arm": best_by_arm.to_dict("records"),
        "arm_summary": arm_summary.to_dict("records"),
        "seed_summary": seed_summary.to_dict("records"),
        "input_registry": input_registry,
        "gmt_path": None if GMT_PATH is None else str(GMT_PATH),
    }

    with open(OUT_META / "gated_cross_modal_attention_lock.json", "w") as f:
        json.dump(final_lock, f, indent=2)

    display(Markdown(f"## Final lock summary written to `{OUT_META / 'gated_cross_modal_attention_lock.json'}`"))
    display(best_by_arm)

    return {
        "mode": mode,
        "out_root": OUT_ROOT,
        "results_dir": OUT_RESULTS,
        "figs_dir": OUT_FIGS,
        "interp_dir": OUT_INTERP,
        "meta_dir": OUT_META,
        "cache_dir": OUT_CACHE,
        "arm_summary_path": OUT_RESULTS / "arm_summary.csv",
        "arm_feature_summary_path": OUT_RESULTS / "arm_feature_summary.csv",
        "best_by_arm_path": OUT_RESULTS / "best_feature_set_by_arm.csv",
    }


def compare_modes(mode_a: str = "pca", mode_b: str = "pathway") -> dict:
    compare_root = MODE_COMPARE_ROOT
    compare_root.mkdir(parents=True, exist_ok=True)

    results_a = NB11_REPORT_ROOT / mode_a / "results"
    results_b = NB11_REPORT_ROOT / mode_b / "results"

    arm_summary_a = pd.read_csv(results_a / "arm_summary.csv")
    arm_summary_b = pd.read_csv(results_b / "arm_summary.csv")
    arm_feature_a = pd.read_csv(results_a / "arm_feature_summary.csv")
    arm_feature_b = pd.read_csv(results_b / "arm_feature_summary.csv")
    best_a = pd.read_csv(results_a / "best_feature_set_by_arm.csv")
    best_b = pd.read_csv(results_b / "best_feature_set_by_arm.csv")

    arm_summary_compare = arm_summary_a.merge(
        arm_summary_b,
        on="arm",
        how="outer",
        suffixes=(f"_{mode_a}", f"_{mode_b}"),
    )
    arm_summary_compare[f"delta_{mode_b}_minus_{mode_a}"] = (
        arm_summary_compare[f"mean_spearman_{mode_b}"] - arm_summary_compare[f"mean_spearman_{mode_a}"]
    )
    arm_summary_compare = arm_summary_compare.sort_values(
        f"delta_{mode_b}_minus_{mode_a}",
        ascending=False,
    ).reset_index(drop=True)
    atomic_to_csv(arm_summary_compare, compare_root / "arm_summary_compare.csv", index=False)

    arm_feature_compare = arm_feature_a.merge(
        arm_feature_b,
        on=["arm", "feature_set"],
        how="outer",
        suffixes=(f"_{mode_a}", f"_{mode_b}"),
    )
    arm_feature_compare[f"delta_{mode_b}_minus_{mode_a}"] = (
        arm_feature_compare[f"mean_spearman_{mode_b}"] - arm_feature_compare[f"mean_spearman_{mode_a}"]
    )
    arm_feature_compare = arm_feature_compare.sort_values(
        ["arm", f"delta_{mode_b}_minus_{mode_a}"],
        ascending=[True, False],
    ).reset_index(drop=True)
    atomic_to_csv(arm_feature_compare, compare_root / "arm_feature_summary_compare.csv", index=False)

    best_a = best_a.copy()
    best_b = best_b.copy()
    best_a["mode"] = mode_a
    best_b["mode"] = mode_b
    best_both = pd.concat([best_a, best_b], ignore_index=True)
    atomic_to_csv(best_both, compare_root / "best_feature_set_by_arm_both_modes.csv", index=False)

    best_mode_by_arm = arm_summary_compare[[
        "arm",
        f"mean_spearman_{mode_a}",
        f"mean_spearman_{mode_b}",
        f"delta_{mode_b}_minus_{mode_a}",
    ]].copy()
    best_mode_by_arm["better_mode"] = np.where(
        best_mode_by_arm[f"delta_{mode_b}_minus_{mode_a}"] > 0,
        mode_b,
        mode_a,
    )
    atomic_to_csv(best_mode_by_arm, compare_root / "best_mode_by_arm.csv", index=False)

    atomic_write_json({
        "mode_a": mode_a,
        "mode_b": mode_b,
        "compare_root": str(compare_root),
        "best_mode_by_arm": best_mode_by_arm.to_dict("records"),
    }, compare_root / "comparison_summary.json")

    plt.figure(figsize=(8, 5))
    x = np.arange(len(arm_summary_compare))
    width = 0.35
    plt.bar(x - width / 2, arm_summary_compare[f"mean_spearman_{mode_a}"], width=width, label=mode_a.upper())
    plt.bar(x + width / 2, arm_summary_compare[f"mean_spearman_{mode_b}"], width=width, label=mode_b.upper())
    plt.xticks(x, arm_summary_compare["arm"], rotation=30, ha="right")
    plt.ylabel("Mean Spearman")
    plt.title("Notebook 11: PCA vs pathway by arm")
    plt.legend()
    plt.tight_layout()
    plt.savefig(compare_root / "arm_summary_pca_vs_pathway.png", dpi=180, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.bar(arm_summary_compare["arm"], arm_summary_compare[f"delta_{mode_b}_minus_{mode_a}"])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel(f"{mode_b} - {mode_a} mean Spearman")
    plt.title("Notebook 11: pathway minus PCA by arm")
    plt.tight_layout()
    plt.savefig(compare_root / "arm_summary_delta_pathway_minus_pca.png", dpi=180, bbox_inches="tight")
    plt.show()

    display(Markdown("## PCA vs pathway comparison"))
    display(arm_summary_compare)
    display(best_mode_by_arm)

    return {
        "arm_summary_compare": arm_summary_compare,
        "arm_feature_compare": arm_feature_compare,
        "best_mode_by_arm": best_mode_by_arm,
        "compare_root": compare_root,
    }


MODE_RUN_OUTPUTS = {}
for mode in RUN_MODES:
    MODE_RUN_OUTPUTS[mode] = run_single_mode(mode)

MODE_COMPARISON = compare_modes("pca", "pathway")

Project root: /home/andrija/Desktop/Final Year Project/FYP
Notebook 11 mode: pca
Notebook 11 output root: /home/andrija/Desktop/Final Year Project/FYP/artifacts/reports/notebook 11/pca
Loaded existing checkpoint rows for mode=pca: 4750

Mode=pca | Seed 19537
    Arm: prot_procan_depmapSanger | eligible_cells=485
      [rna] drug 1/10 | IXAZOMIB (BRD:BRD-K78659596-001-03-9) | n_cells=272 | GroupKFold(n_splits=10)
      [rna] drug 5/10 | A-1065-5 (BRD:BRD-K87124298-001-03-4) | n_cells=270 | GroupKFold(n_splits=10)
      [rna] drug 10/10 | PI3K-IN-2 (BRD:BRD-K62374002-001-01-5) | n_cells=268 | GroupKFold(n_splits=10)
        [mid-checkpoint] drug 10/10
      [cnv] drug 1/10 | IXAZOMIB (BRD:BRD-K78659596-001-03-9) | n_cells=272 | GroupKFold(n_splits=10)
      [cnv] drug 5/10 | A-1065-5 (BRD:BRD-K87124298-001-03-4) | n_cells=270 | GroupKFold(n_splits=10)
      [cnv] drug 10/10 | PI3K-IN-2 (BRD:BRD-K62374002-001-01-5) | n_cells=268 | GroupKFold(n_splits=10)
        [mid-checkpoint] drug 10/1

KeyboardInterrupt: 